In [1]:
from alerce.core import Alerce
import pandas as pd
alerce = Alerce()

In [28]:
classifiers = alerce.query_classifiers(format = 'pandas',
                                       survey = 'ztf')
classifier_names = classifiers["classifier_name"].unique()
classifier_names

<StringArray>
[                  'lc_classifier_transient',
                  'lc_classifier_stochastic',
                    'lc_classifier_periodic',
      'LC_classifier_ATAT_forced_phot(beta)',
      'LC_classifier_BHRF_forced_phot(beta)',
                             'lc_classifier',
                         'lc_classifier_top',
                          'stamp_classifier',
            'lc_classifier_BHRF_forced_phot',
        'lc_classifier_BHRF_forced_phot_top',
   'lc_classifier_BHRF_forced_phot_periodic',
 'lc_classifier_BHRF_forced_phot_stochastic',
  'lc_classifier_BHRF_forced_phot_transient',
                'stamp_classifier_2025_beta']
Length: 14, dtype: str

In [21]:
for each in classifiers:
    if "SNII" in each["classes"]:
        print(each["classifier_name"])

lc_classifier_transient
LC_classifier_ATAT_forced_phot(beta)
LC_classifier_BHRF_forced_phot(beta)
lc_classifier
lc_classifier_BHRF_forced_phot
lc_classifier_BHRF_forced_phot_transient


In [17]:
params = {
   "classifier": "LC_classifier_ATAT_forced_phot(beta)",
   "class_name": "SNIIb",
   "probability": "0.5",
   "format": "pandas"
}

page = 1
all_data = []

while True:
   try:
      print(f"Querying page: {page}")
      dataframe = alerce.query_objects(**params,
                                       pagesize = 1000,
                                       page = page)
      if len(dataframe) == 0:
         print("Done")
         break

      all_data.append(dataframe)
      page = page + 1

   except Exception as error:
      print(error)
      break

# sn2 = pd.concat(all_data, ignore_index = True)

Querying page: 1
Querying page: 2
Querying page: 3
Querying page: 4
Querying page: 5
Querying page: 6


KeyboardInterrupt: 

In [35]:
params ={"survey": "ztf",
         "probability": 0.48,
         "format": "pandas"}

classifier_names = ["lc_classifier_transient", 
                    "LC_classifier_ATAT_forced_phot(beta)",
                    "LC_classifier_BHRF_forced_phot(beta)",
                    "lc_classifier",
                    "lc_classifier_BHRF_forced_phot",
                    "lc_classifier_BHRF_forced_phot_transient"]

class_name = ["SNII", "SNIIb"]

all_data = []

# Run through each classifier, in each classifier, run through SNII and SNIIb
for each_classifier in classifier_names:

   for each_class in class_name:

      page = 1

      while True:
         try:
            print(f"Querying {each_classifier}, {each_class}, page {page}")

            dataframe = alerce.query_objects(**params,
                                             classifier = each_classifier,
                                             class_name = each_class,
                                             pagesize = 100,
                                             page = page)
            if len(dataframe) == 0:
               print("Done")
               break

            all_data.append(dataframe)
            page = page + 1

         except Exception as error:
            print(f"{each_classifier}, {each_class} has error {error}")
            break
    
# sn2 = pd.concat(all_data, ignore_index = True)

Querying lc_classifier_transient, SNII, page 1
Querying lc_classifier_transient, SNII, page 2
Querying lc_classifier_transient, SNII, page 3
Querying lc_classifier_transient, SNII, page 4
Querying lc_classifier_transient, SNII, page 5
Querying lc_classifier_transient, SNII, page 6
Querying lc_classifier_transient, SNII, page 7
Querying lc_classifier_transient, SNII, page 8
Querying lc_classifier_transient, SNII, page 9
Querying lc_classifier_transient, SNII, page 10
Querying lc_classifier_transient, SNII, page 11
Querying lc_classifier_transient, SNII, page 12
Querying lc_classifier_transient, SNII, page 13
Querying lc_classifier_transient, SNII, page 14
Querying lc_classifier_transient, SNII, page 15
Querying lc_classifier_transient, SNII, page 16
Querying lc_classifier_transient, SNII, page 17
Querying lc_classifier_transient, SNII, page 18
Querying lc_classifier_transient, SNII, page 19
Querying lc_classifier_transient, SNII, page 20
Querying lc_classifier_transient, SNII, page 21
Q

In [37]:
sn2 = pd.concat(all_data, ignore_index = True)
dataframe = pd.DataFrame(sn2)
dataframe.to_csv("ZTF_new.csv", index = False)

In [2]:
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import astropy.units as u
from astropy import coordinates
from astropy.io.votable import parse_single_table
from astropy.time import Time
from astropy.table import Table, Column
from astropy.coordinates import Distance
from astropy.cosmology import WMAP7
%pip install pyvo
import pyvo

Note: you may need to restart the kernel to use updated packages.


In [23]:
# Query ZTF using ADQL

TAP_URL = "https://tap.alerce.online/tap"
tap_service = pyvo.dal.TAPService(TAP_URL)

def tap_query(query, maxrec=10_000_000):
    result = tap_service.search(query, maxrec=maxrec, language = "ADQL")
    return result.to_table().to_pandas()

def tap_query_idlist(query, index, maxrec=10_000_000):
    # Upload oid (or other identifier fields) list as a TAP table.
    # IMPORTANT: call the upload column something other than "oid", as "oid" collides
    # with PostgreSQL's system column and breaks the join ("objectid" works)
    col = 'objectid'
    oid_table = Table({col: index})
    result = tap_service.search(query, uploads={'mylist': oid_table},
                                maxrec=maxrec).to_table().to_pandas()
    if col in result.columns:
        result.drop(columns=[col], inplace=True)
    return result

In [24]:
query = '''
SELECT
    o.oid, o.meanra, o.meandec, o.ndet,
    o.firstmjd, o.deltajd, o.g_r_max,
    p.classifier_name, p.class_name,
    p.classifier_version,
    p.ranking, p.probability
FROM
    ztf.object AS o INNER JOIN ztf.probability AS p
    ON o.oid = p.oid
WHERE
    p.classifier_name IN ('lc_classifier_transient', 'LC_classifier_ATAT_forced_phot(beta)', 'LC_classifier_BHRF_forced_phot(beta)', 'lc_classifier',
    'lc_classifier_BHRF_forced_phot', 'lc_classifier_BHRF_forced_phot_transient')
    AND o.oid IN
(
SELECT
    oid
FROM
    ztf.probability
WHERE
    classifier_name IN ('lc_classifier_transient', 'LC_classifier_ATAT_forced_phot(beta)', 'LC_classifier_BHRF_forced_phot(beta)', 'lc_classifier',
    'lc_classifier_BHRF_forced_phot', 'lc_classifier_BHRF_forced_phot_transient')
    AND class_name IN ('SNII','SNIIb')
    AND ranking = 1
    AND probability > 0.9
)
'''
SNe = tap_query(query)
print(SNe.shape)
SNe.set_index('oid', inplace=True)
SNe.head()

DALServiceError: 502 Server Error: Bad Gateway for url: https://tap.alerce.online/tap/sync

In [12]:
SNe.classifier_version.value_counts()

classifier_version
0.2.0    155254
0.3.1     22858
1.0.0     18354
0.3.4     11836
0.3.6      7018
Name: count, dtype: int64

In [13]:
# SNe that have two classification versions
repeated = set(SNe.loc[SNe.classifier_version == "hierarchical_rf_1.1.0"].index.unique()).\
intersection(SNe.loc[SNe.classifier_version == "lc_classifier_1.1.13"].index.unique())

In [14]:
SNe = pd.concat([SNe.loc[SNe.classifier_version == "lc_classifier_1.1.13"], 
                 SNe.loc[~SNe.index.isin(repeated) \
                 & (SNe.classifier_version=="hierarchical_rf_1.1.0")]])

In [15]:
SNe

,meanra,meandec,ndet,firstmjd,deltajd,g_r_max,classifier_name,class_name,classifier_version,ranking,probability
oid,,,,,,,,,,,
